# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/forhadmia231/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

One row represents one content page for one client on one report date.

For this assignment, I use the March 2026 panel
(month = '2026-03') as the analysis window. This is a mid-panel
month, so I can use it to inspect and develop features without using
the final June 2026 month for label development.

## 1. Unit of analysis + time window

One row represents one content page's search performance observation for the selected analysis window.

For this assignment, I will use March 2026 (month = '2026-03') as my mid-panel analysis month. I chose a mid-panel month instead of the final month so that I can inspect the available signals without using the final outcome month for label development.

Features:
I will use five signals that are available at the decision moment:
gsc_impressions, gsc_clicks, gsc_avg_position, ga4_engaged_sessions,
and scroll_events.

Label or ranking target:
The goal is to rank content pages by refresh priority. The exact
priority label will be defined using outcome information separately
from the decision-time features.

Context:
The unit of analysis is content performance for a specific client and
report date. The decision is which content pages an SEO team should
prioritize for review or refresh.

Excluded:
I deliberately exclude future outcome information from the feature set
because it would not be available when the refresh decision is made.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")

print("Token loaded:", HF_TOKEN is not None)

Token loaded: True


## 2. Fields: feature / label / context / excluded

Feature:
I will use search performance signals that are available at the time of the refresh decision, including impressions, clicks, CTR, average position, and search demand.

Label or ranking target:
I want to rank content pages by their estimated priority for refresh. The exact proxy will be based on observed performance signals and later outcome data, rather than claiming that the current metrics alone prove a page will improve.

Context:
The analysis focuses on SEO content performance and the decision of which pages an SEO team should review or refresh first.

Excluded:
I deliberately exclude future outcome information from the feature set because it would not be available when the SEO team makes the refresh decision.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from huggingface_hub import HfApi

api = HfApi(token=HF_TOKEN)

files = api.list_repo_files(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset"
)

for file in files[:30]:
    print(file)

.gitattributes
README.md
dim_clients.parquet
dim_content.parquet
fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_performance/month=2026-03/data_0.parquet
fact_content_daily_performance/mont

In [4]:
from huggingface_hub import hf_hub_download
import pandas as pd

file_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    token=HF_TOKEN
)

df = pd.read_parquet(file_path)

print(df.shape)
print(df.columns.tolist())

df.head()

print(df.shape)
print(df.columns.tolist())

(9841378, 30)
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']
(9841378, 30)
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 

## 3. Verify it with queries (grain, counts, missing values, windows)

Verification results:

1. Grain:
The maximum number of rows for the same report_date, client_hash_id,
and content_hash_id combination was 1. This supports the contract that
one row represents one content page for one client on one report date.

2. Slice size and date span:
The March 2026 slice contains 9,841,378 rows. The report_date range is
from 2026-03-01 to 2026-03-31.

3. Availability:
Out of 9,841,378 total rows, 3,611,061 rows have
gsc_data_available IS TRUE. This means GSC-based features can only be
used for the subset where that source was available.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Query 1: Verify the grain

grain_check = (
    df.groupby(["report_date", "client_hash_id", "content_hash_id"])
      .size()
      .reset_index(name="rows")
)

print("Maximum rows for one date-client-content combination:",
      grain_check["rows"].max())

grain_check.head()

Maximum rows for one date-client-content combination: 1


,report_date,client_hash_id,content_hash_id,rows
0,2026-03-01,client_0797ff3a1fc9a6a5,content_004e9c4c32e88631,1
1,2026-03-01,client_0797ff3a1fc9a6a5,content_0236ef736698e17c,1
2,2026-03-01,client_0797ff3a1fc9a6a5,content_025f6cfd3c298870,1
3,2026-03-01,client_0797ff3a1fc9a6a5,content_0263d5f9b7a2ecd4,1
4,2026-03-01,client_0797ff3a1fc9a6a5,content_02752c6c1c60161f,1


In [6]:
# Query 2: Row count and date span

print("Row count:", len(df))
print("Start date:", df["report_date"].min())
print("End date:", df["report_date"].max())

Row count: 9841378
Start date: 2026-03-01
End date: 2026-03-31


In [7]:
# Query 3: Availability check

available_rows = df.query("gsc_data_available == True")

print("Rows before availability filter:", len(df))
print("Rows where gsc_data_available is TRUE:", len(available_rows))

Rows before availability filter: 9841378
Rows where gsc_data_available is TRUE: 3611061


In [8]:
import duckdb

result = duckdb.sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows
    FROM df
""").df()

result

,total_rows,gsc_available_rows
0,9841378,3611061


In [9]:
grain_check = df.groupby(
    ["report_date", "client_hash_id", "content_hash_id"]
).size()

print("Maximum rows for one date-client-content combination:", grain_check.max())

Maximum rows for one date-client-content combination: 1


In [10]:
features_df = df.loc[
    df["gsc_data_available"] == True,
    [
        "report_date",
        "client_hash_id",
        "content_hash_id",
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
        "ga4_engaged_sessions",
        "scroll_events"
    ]
].copy()

features_df.head()

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_engaged_sessions,scroll_events
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,0,3.350000,NaN,NaN
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0,0.000000,NaN,NaN
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,1,4.928000,NaN,NaN
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,0,4.000000,NaN,NaN
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,0,2.272727,NaN,NaN


Feature 1 — gsc_impressions
Knowable at the decision moment because it records search impressions
already observed before the refresh decision.

Feature 2 — gsc_clicks
Knowable at the decision moment because historical search clicks are
already available from previous reporting.

Feature 3 — gsc_avg_position
Knowable at the decision moment because the page's historical average
search position can be observed before deciding whether to refresh it.

Feature 4 — ga4_engaged_sessions
Knowable at the decision moment because engagement from previous user
sessions has already occurred and can be measured.

Feature 5 — scroll_events
Knowable at the decision moment because historical scrolling behavior
has already been recorded before the refresh decision.

In [11]:
import duckdb

availability_check = duckdb.sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS available_rows
    FROM df
""").df()

availability_check

,total_rows,available_rows
0,9841378,3611061


In [12]:
grain_check = df.groupby(
    ["report_date", "client_hash_id", "content_hash_id"]
).size()

print("Maximum rows for one date-client-content combination:", grain_check.max())

Maximum rows for one date-client-content combination: 1


In [13]:
features_df = df.loc[
    (df["gsc_data_available"] == True) &
    (df["ga4_data_available"] == True),
    [
        "report_date",
        "client_hash_id",
        "content_hash_id",
        "gsc_impressions",
        "gsc_avg_position",
        "ga4_sessions",
        "ga4_engaged_sessions",
        "scroll_events",
        "gsc_clicks"
    ]
].copy()

print("Feature frame shape:", features_df.shape)

features_df.head()

Feature frame shape: (364347, 9)


,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_avg_position,ga4_sessions,ga4_engaged_sessions,scroll_events,gsc_clicks
14480,2026-03-01,client_65de48885f4ef01b,content_5c80451459c29b4a,5,5.400000,1.0,0.0,0.0,0
14748,2026-03-01,client_65de48885f4ef01b,content_b1f61fc81b28b2d4,39,5.666667,2.0,0.0,0.0,0
14768,2026-03-01,client_65de48885f4ef01b,content_e25ea7297a1dffd3,179,5.156425,2.0,0.0,0.0,0
14783,2026-03-01,client_65de48885f4ef01b,content_6b0149a80607dac3,72,7.694444,1.0,0.0,0.0,0
14855,2026-03-01,client_65de48885f4ef01b,content_62673eea26c31c17,3282,6.167885,1.0,0.0,0.0,1


**Feature Explaination**

1. gsc_impressions
Knowable at the decision moment because historical search impressions
have already been recorded before the refresh decision.

2. gsc_avg_position
Knowable at the decision moment because the page's observed search
position is available from previous search performance.

3. ga4_sessions
Knowable at the decision moment because historical user sessions have
already occurred before the decision.

4. ga4_engaged_sessions
Knowable at the decision moment because engagement from previous
sessions has already been recorded.

5. scroll_events
Knowable at the decision moment because historical user scrolling
behavior has already been observed.

For this notebook, I keep the feature set deliberately small. These
features are signals available from observed search and analytics
performance rather than future outcome information.

**Leakage Trap**


In [14]:
sample_df = features_df.sample(
    n=100000,
    random_state=42
).copy()

print(sample_df.shape)

(100000, 9)



**Temporary Proxy Level Make**



In [15]:
median_clicks = sample_df["gsc_clicks"].median()

sample_df["high_clicks"] = (
    sample_df["gsc_clicks"] > median_clicks
).astype(int)

print("Median clicks:", median_clicks)
print(sample_df["high_clicks"].value_counts(normalize=True))

Median clicks: 0.0
high_clicks
0    0.50258
1    0.49742
Name: proportion, dtype: float64


For the leakage demonstration only, I use a temporary proxy label:
whether a row has above-median GSC clicks. This is not my final
refresh-priority label. It is only used to demonstrate how a
label-derived feature can create misleadingly high model performance.

In [17]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

feature_cols = [
    "gsc_impressions",
    "gsc_avg_position",
    "ga4_sessions",
    "ga4_engaged_sessions",
    "scroll_events"
]

X = sample_df[feature_cols]
y = sample_df["high_clicks"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

predictions = model.predict(X_test)

honest_score = accuracy_score(y_test, predictions)

print(" accuracy:", honest_score)

 accuracy: 0.6781


In [18]:
# DELIBERATE LEAKAGE EXPERIMENT
# Adding the label itself as a feature on purpose

X_leaky = sample_df[
    feature_cols + ["high_clicks"]
]

X_train_leaky, X_test_leaky, y_train_leaky, y_test_leaky = train_test_split(
    X_leaky,
    y,
    test_size=0.2,
    random_state=42
)

leaky_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

leaky_model.fit(X_train_leaky, y_train_leaky)

leaky_predictions = leaky_model.predict(X_test_leaky)

leaky_score = accuracy_score(
    y_test_leaky,
    leaky_predictions
)

print("Leaky accuracy:", leaky_score)

Leaky accuracy: 1.0


In [20]:
# Remove the leaked label-derived column

X_honest = sample_df[feature_cols].copy()

print("Final  features:")
print(X_honest.columns.tolist())

print("\nFinal  accuracy:", honest_score)

Final  features:
['gsc_impressions', 'gsc_avg_position', 'ga4_sessions', 'ga4_engaged_sessions', 'scroll_events']

Final  accuracy: 0.6781


### Deliberate Leakage Experiment

Using only the five decision-time features, the quick model achieved an
accuracy of **0.6781 (67.81%)**.

For the leakage experiment, I deliberately added `high_clicks` as an
input feature. This column is derived directly from the target label,
so it gives the model access to the answer.

The leaky model achieved an accuracy of **1.0 (100%)**. This near-perfect
score does not represent genuine predictive ability. It happened because
the answer was included in the input data.

I removed the leaked column after the experiment and kept **0.6781
(67.81%)** as the honest score.

This experiment demonstrates that a very high model score can be
misleading when label-derived or future information leaks into the
feature set.

## 4. Data Limits

### Limitation

This analysis uses only the March 2026 slice of the warehouse. Therefore,
the patterns observed here may not represent performance across other
months.

In addition, not every row has GSC and GA4 data available. After applying
the availability filter, only the rows with the required data sources
were used for the feature frame. This may exclude some clients or content
pages from the analysis.

Another limitation is that `high_clicks` is only a temporary proxy label
used to demonstrate data leakage. It is not the final definition of
content refresh priority.

Future work should define the refresh-priority label using a proper
past-to-future outcome window and validate the model on a separate,
sealed time period.

## Self-check

## 5. Self-check

- [x] I defined what one row represents.
- [x] I identified the table used for my lane.
- [x] I used March 2026 as the analysis window.
- [x] I stated what I want to rank or predict.
- [x] I deliberately excluded future or label-derived information.
- [x] I verified the grain of the data.
- [x] I checked the row count and date span.
- [x] I checked data availability using `IS TRUE`.
- [x] I created exactly five decision-time features.
- [x] I explained why each feature is available at the decision moment.
- [x] I performed a deliberate data leakage experiment.
- [x] I removed the leaked feature.
- [x] I kept the honest score rather than the leaky score.
- [x] I documented a limitation of the data slice.